In [ ]:
#Import packages
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Descriptors
import pandas as pd

In [37]:
# User inputs
Ngen = 3  # Number of generations
non_exhaustive = True
#reaction_library = pd.DataFrame({
#    'Scheme_name': ['Hydrogenolysis', 'Vicinal Dehalogenation'],
#    'Reaction_expression': ['[#6;A:1][#17,#35,#53]>>[#6;A:1]', '[#17,#35,#53][#6;A:1][#6;A:2][#17,#35,#53]>>[#6;A:1]=[#6;A:2]'],
#    'Rank': [4, 4],
#    'Reactivity_rule': [True, True],  # Not needed
#    'Selectivity_rule': [False, False]  # Simplified for demonstration
#})
# reaction_library = pd.read_csv('AbioticReductionLibrary_v1.5.txt', sep='\t')
reaction_library = pd.read_csv('AbioticHydrolysisLibrary_v1.8.txt', sep='\t')
Scheme_name = reaction_library.iloc[:,0]
Reaction_expression = reaction_library.iloc[:,1]
Rank = reaction_library.iloc[:,2]
Selectivity_rule = reaction_library.iloc[:,3]

In [42]:
# Initialize parent molecule
mol = ['COC(=O)CC(NC(=O)[C@@H](NC(=O)OC(C)C)C(C)C)c1ccc(Cl)cc1']  # Parent SMILES
gen = [0]
ParentID = [0]
F = 0.0
F_deg = [0.0]
route = ['parent']
production = [1.0]
accumulation = [0.0]
N_prod = list()
N_prod = [1]
mol_index = 0

# Initialize reaction objects
reactions = [AllChem.ReactionFromSmarts(exp) for exp in reaction_library['Reaction_expression']]

In [43]:
# Main loop for generations.  To calculate accumulation in the final generation, it is necessary
# to predict an extra generation of products beyond the number of generations requested by the user.
gen_count = 0
for i in range(Ngen):
    gen_count += 1
#    print("i = ", i, gen_count)
    Np = 0 
    # Loop over products in previous generation as parents in current generation
#    print("number of products in prev gen =",N_prod)
    for p in range(N_prod[i]):
        j = mol_index + p
        # Initiate F_deg, the sum of the formation values for all schemes generating unique product pairs        
        if j >= len(F_deg):
            F_deg.append(0.0)
        else:
            F_deg[j] = 0.0
        # Assign the parent ID for bookkeeping
        parent_idx = j
#        print("parent ID = ",parent_idx)
#        print(mol)
        reactant_mol = Chem.MolFromSmiles(mol[j])
        parent_check = Chem.MolToSmiles(reactant_mol)
        print("parent = ",parent_check)
        # create empty arrays to store SMILES, formation values, and scheme names for unique products
        smiles_list = list()
        prod_form = list()
        prod_scheme = list()
        max_prod_form = 7.0
        # loop over schemes in the reaction library
        for k in range(len(reactions)):
            reaction = reactions[k]
            F = 7**Rank[k]
            num_exp_products = reaction.GetNumProductTemplates()
#            print("Scheme =", Scheme_name[k], num_exp_products)
#            print(F)
            num_prod_set = 0
            uniq_prod_set = set()
            product_tuple = ()
            # Reactions in Metabolizer libraries have only one reactant; define single element tuple for this reactant
            reactant_tuple = (reactant_mol,)
            try:
                products = reaction.RunReactants(reactant_tuple)            
                for product_set in products:
#               RunReactants generates duplicate sets of products at each possible reactant site. 
#               Remove these duplicates (as well as extra replicates that can occur when the molecule 
#               has more than one equivalent reaction sites). 
                    max_prod_form = max(max_prod_form,F)
                    if num_exp_products == 1:
                        ps1 = Chem.MolToSmiles(products[num_prod_set][0])
                        product_tuple = (ps1,)                
                    else:  
                        ps1 = Chem.MolToSmiles(products[num_prod_set][0])
                        ps2 = Chem.MolToSmiles(products[num_prod_set][1])
                        moldummy = Chem.MolFromSmiles(ps1)
                        mw1 = Descriptors.MolWt(moldummy)
                        moldummy = Chem.MolFromSmiles(ps2)
                        mw2 = Descriptors.MolWt(moldummy)
                        if mw2 > mw1:
                            product_tuple = (ps2, ps1)
                        else:
                            product_tuple = (ps1, ps2)
#               Only retain the unique products or product pairs generated by RunReactants.
#                    print(product_tuple)
#                    print(F)
#                    print(smiles_list)
                    if product_tuple not in uniq_prod_set:
                        uniq_prod_set.add(product_tuple)
#                        print("here's unique",uniq_prod_set)
                        F_deg[parent_idx] += F
#                        print("Parent sum of F",F_deg)
#                   # Extend lists for first (or only) product
                        smiles_list.append(ps1)
                        prod_form.append(F)
                        prod_scheme.append(Scheme_name[k])
#                   # Extend lists for second product if the reaction involves cleavage
                        if num_exp_products == 2 and ps1 != ps2:
                            smiles_list.append(ps2)
                            prod_form.append(F)
                            prod_scheme.append(Scheme_name[k])
#                    print("list with product =",smiles_list)
#               # Increase the num_prod_set counter, which is the number of products 
#                 (for num_exp_products = 1) or product pairs (for num_exp_products = 2)
#                    num_prod_set += 1
#                    print(uniq_prod_set)
            except Exception:
                continue
        # End loop over schemes in the reaction library
        # Default (non-exhaustive): only retain products if their formation is 10% or more of the
        # maximum formation value for all products formed from parent p in generation i
        # Extend lists for products formed from parent p in generation i
        print(max_prod_form)
        print(prod_form)
        if non_exhaustive:
            for id_prod in range(len(prod_form)):
                if prod_form[id_prod]>=0.1*max_prod_form:
                    Np += 1
                    mol.append(smiles_list[id_prod])
                    gen.append(gen_count)
                    ParentID.append(parent_idx)
                    route.append(prod_scheme[id_prod])     # Store scheme name
                    production.append(prod_form[id_prod])   # Store formation for production calculation
                    accumulation.append(0.0)  # Placeholder for accumulation calculation
        else:
            for id_prod in range(len(prod_form)):
                Np += 1
                mol.append(smiles_list[id_prod])
                gen.append(gen_count)
                ParentID.append(parent_idx)
                route.append(prod_scheme[id_prod])     # Store scheme name 
                production.append(prod_form[id_prod])   # Store formation for production calculation
                accumulation.append(0.0)  # Placeholder for accumulation calculation

    # End loop over parents in current generation
    N_prod.append(Np)
    mol_index += N_prod[i]
    prod_index = mol_index + Np
    print(production)
    print(accumulation)

parent =  COC(=O)CC(NC(=O)[C@@H](NC(=O)OC(C)C)C(C)C)c1ccc(Cl)cc1
343
[np.int64(49), np.int64(49), np.int64(7), np.int64(7), np.int64(343), np.int64(343)]
[1.0, np.int64(49), np.int64(49), np.int64(343), np.int64(343)]
[0.0, 0.0, 0.0, 0.0, 0.0]
parent =  CC(C)OC(=O)N[C@H](C(=O)NC(CC(=O)O)c1ccc(Cl)cc1)C(C)C
343
[np.int64(7), np.int64(7), np.int64(343), np.int64(343)]
parent =  CO
7.0
[]
parent =  COC(=O)CC(NC(=O)[C@@H](N)C(C)C)c1ccc(Cl)cc1
49
[np.int64(49), np.int64(49), np.int64(7), np.int64(7)]
parent =  CC(C)O
7.0
[]
[1.0, np.int64(49), np.int64(49), np.int64(343), np.int64(343), np.int64(343), np.int64(343), np.int64(49), np.int64(49), np.int64(7), np.int64(7)]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
parent =  CC(C)[C@H](N)C(=O)NC(CC(=O)O)c1ccc(Cl)cc1
7.0
[np.int64(7), np.int64(7)]
parent =  CC(C)O
7.0
[]
parent =  CC(C)[C@H](N)C(=O)NC(CC(=O)O)c1ccc(Cl)cc1
7.0
[np.int64(7), np.int64(7)]
parent =  CO
7.0
[]
parent =  CC(C)[C@H](N)C(=O)O
7.0
[]
parent =  COC(=O)CC(N)c1c

[15:43:48] mapped atoms in the reactants were not mapped in the products.
  unmapped numbers are: 2 
[15:43:48] mapped atoms in the reactants were not mapped in the products.
  unmapped numbers are: 4 
[15:43:48] mapped atoms in the reactants were not mapped in the products.
  unmapped numbers are: 3 
[15:43:48] mapped atoms in the reactants were not mapped in the products.
  unmapped numbers are: 3 
[15:43:48] mapped atoms in the reactants were not mapped in the products.
  unmapped numbers are: 5 
[15:43:48] mapped atoms in the reactants were not mapped in the products.
  unmapped numbers are: 5 


In [ ]:
# Calculate production values
print(production)
print(F_deg)
print(ParentID)

# Calculate accumulation values
print(accumulation)
for j in range(1, mol_index):
    print(j,prod_index)
    print(production[j])
    print(ParentID[j])
    print(F_deg[j])
    form_temp = production[j]
    parent_deg = (production[ParentID[j]]-accumulation[ParentID[j]])/F_deg[ParentID[j]]
    production[j] = parent_deg*form_temp
    accumulation[j] = parent_acc*(form_temp-F_deg[j]/F_deg[ParentID[j]])

# Output results
for idx in range(0, prod_index):
    print("Molecule:", idx, "SMILES=", mol[idx], "Generation=", gen[idx], "Production=", production[idx], "Accumulation=", accumulation[idx], route[idx])

[1.0, np.int64(49), np.int64(49), np.int64(343), np.int64(343), np.int64(343), np.int64(343), np.int64(49), np.int64(49), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(49), np.int64(49)]
[np.float64(399.0), np.float64(350.0), 0.0, np.float64(56.0), 0.0, np.float64(7.0), 0.0, np.float64(7.0), 0.0, 0.0, np.float64(49.0)]
[0, 0, 0, 0, 0, 1, 1, 3, 3, 3, 3, 5, 5, 7, 7, 10, 10]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
1 17
49
0
350.0
2 17
49
0
0.0
3 17
343
0
56.0
4 17
343
0
0.0
5 17
343
1
7.0
6 17
343
1
0.0
7 17
49
3
7.0
8 17
49
3
0.0
9 17
7
3
0.0
10 17
7
3
49.0
Molecule: 0 SMILES= COC(=O)CC(NC(=O)[C@@H](NC(=O)OC(C)C)C(C)C)c1ccc(Cl)cc1 Generation= 0 Production= 1.0 Accumulation= 0.0 parent
Molecule: 1 SMILES= CC(C)OC(=O)N[C@H](C(=O)NC(CC(=O)O)c1ccc(Cl)cc1)C(C)C Generation= 1 Production= 0.12280701754385964 Accumulation= 0.12060853889108737 Carboxylic Acid Ester Hydrolysis
Molecule: 2 SMILES= CO Generation= 